# Stage 3 on real sessions (Mkechinov or RetailRocket)

**Goal.** Run the semantic approaches (GSDMM, LDA, SBERT + k-means, optional HDBSCAN) on real sessions and compare them with Stage 2 **without ground truth**. The controls notebook (`semantic_controls.ipynb`) says what each method *can* do; this one shows what it *does* on your data.

### Why the evaluation looks different from the controls
There is no true grouping, so ARI against the truth does not exist. Following `evaluation_protocol.md`:

| Question | Tool | Why it is trustworthy |
|---|---|---|
| Are the clusters better than chance? | **random-partition null** (same cluster sizes, shuffled labels) | a statistic that real clusters do not beat random partitions on is not a finding |
| Do the clusters mean something? | **predictive validity**: remove the purchase events from every session *before* clustering, then ask how much of "did this session end in a purchase" the clusters explain | using the full session would be circular, because the outcome is derived from the events being clustered |
| Which clustering is more separated? | **silhouette in a common reference space** (k-gram cosine, on one subsample) next to each method's own | each method's own silhouette lives in a different geometry and cannot be ranked |
| Is it stable? | ARI between two runs with different seeds | a clustering that changes with the seed describes the seed |
| Do the approaches agree? | ARI **between** approaches | agreement, not quality |

### Why the token format matters here
`enriched_events` tokens look like `view_electronics.smartphone`. Unlike the three bare event types of RetailRocket, the category path is made of real words, so a language model has something to read. This is where the hypothesis "pre-trained semantics help" can actually be tested.

In [ ]:
import time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from journey_analyzer import TransitionModel, SemanticNLPModel
from journey_analyzer.evaluation import adjusted_rand_index
from journey_analyzer.semantic import SessionRenderer, SBERTEncoder, HashingEncoder
from journey_analyzer.transitions import START, END
from journey_analyzer.visualization import plot_transition_matrix

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.width", 200); pd.set_option("display.max_colwidth", 110)

## 1. Configuration
One cell to edit. `N_SAMPLE` is the number of sessions used by the clustering experiments: the pairwise-free methods could take more, but GSDMM and LDA are the slow ones (section 9 measures it) and the evaluation protocol uses about 5,000-20,000 for quality comparisons.

In [ ]:
DATASET = "mkechinov"                 # "mkechinov" or "retailrocket"
CONFIG = {
    "mkechinov":    dict(path="../../data/processed/mkechinov_sessions.parquet",    col="enriched_events"),
    "retailrocket": dict(path="../../data/processed/retailrocket_sessions.parquet", col="events"),
}[DATASET]
N_SAMPLE = 20_000                     # sessions used for the clustering experiments
K = 6                                 # clusters shown in detail (section 5 sweeps k = 2..10)
K_UB = 15                             # GSDMM upper bound
SEED = 0
USE_SBERT = True
SBERT_MODEL = "sentence-transformers/all-MiniLM-L6-v2"   # or a local folder

In [ ]:
def make_encoder():
    if USE_SBERT:
        try:
            enc = SBERTEncoder(SBERT_MODEL); enc.load()
            return enc, "SBERT"
        except Exception as e:
            print("!! SBERT not available -> using the lexical STAND-IN.\n   reason:", str(e).splitlines()[0][:160])
    return SBERTEncoder(encoder=HashingEncoder(384)), "HashingEncoder (STAND-IN, no semantics)"

ENCODER, ENCODER_NAME = make_encoder()
print("encoder:", ENCODER_NAME, "| one-off load time: %.1fs" % ENCODER.load_seconds_)

## 2. Load, filter, sample
Same filter as the Stage 2 notebooks (at least 2 events, up to the 99th length percentile). Only the event column is read. **Pitfall:** the file is sorted by user, so the first rows are not a random sample; the sample below is random.

In [ ]:
df = pd.read_parquet(CONFIG["path"], columns=[CONFIG["col"]])
seqs = df[CONFIG["col"]].apply(lambda x: [str(e).strip().lower() for e in x])
del df
lens = seqs.str.len()
keep = (lens >= 2) & (lens <= np.percentile(lens, 99))
pool = seqs[keep].tolist()
del seqs
print(f"{int(keep.sum()):,} sessions after the filter (of {len(lens):,})")

rng = np.random.default_rng(SEED)
idx = rng.choice(len(pool), size=min(N_SAMPLE, len(pool)), replace=False)
sample = [pool[i] for i in idx]
print(f"sample: {len(sample):,} sessions | mean length {np.mean([len(s) for s in sample]):.1f} | "
      f"distinct sessions {len(set(map(tuple, sample))) / len(sample):.0%} | distinct event tokens {len({e for s in sample for e in s}):,}")
print("example:", sample[0][:6])

## 3. What the encoder will read
Rendering is a design variable (see the controls notebook). The table shows the same session under three renderings, and `truncation_report` checks the input limit: text beyond it is **cut at the end**, which is where purchases are. The word-count fallback is a lower bound for word pieces.

In [ ]:
s = max(sample[:200], key=len)
RENDER = {"natural": SessionRenderer("natural"), "natural + RLE": SessionRenderer("natural", rle=True),
          "natural + RLE + last 10": SessionRenderer("natural", rle=True, window=10)}
display(pd.Series({k: r.render(s)[:230] for k, r in RENDER.items()}, name=f"one session of length {len(s)}").to_frame())
rows = []
for k_, r in RENDER.items():
    rep = ENCODER.truncation_report(r.render_many(sample[:5000]))
    rows.append({"rendering": k_, "p50": rep["p50"], "p99": rep["p99"], "max": rep["max"], "limit": rep["limit"],
                 "share over limit": f"{rep['share_over_limit']:.2%}", "unit": rep["unit"]})
pd.DataFrame(rows).set_index("rendering")

## 4. Outcome for predictive validity
`converted` = the session contains a purchase (`purchase_*` in Mkechinov, `transaction` in RetailRocket). **The purchase events are then removed from the sessions that are clustered**, so the clusters cannot see the outcome directly; `cart` / `addtocart` events stay (they are legitimate behaviour, and strongly predictive, which is the point of the test).

In [ ]:
PURCHASE = {"purchase", "transaction"}
r0 = SessionRenderer()
is_purchase = lambda e: r0.split_token(e)[0] in PURCHASE

converted_all = np.array([any(is_purchase(e) for e in s) for s in sample])
stripped = [[e for e in s if not is_purchase(e)] for s in sample]
ok = np.array([len(s) >= 2 for s in stripped])                 # clustering needs some behaviour left
X = [s for s, k_ in zip(stripped, ok) if k_]
y = converted_all[ok].astype(float)
print(f"{len(X):,} sessions kept for clustering ({(~ok).sum():,} dropped: fewer than 2 events once purchases are removed)")
print(f"conversion rate in the clustered sample: {y.mean():.2%}")

## 5. Choosing k for the k-means routes (SBERT / embeddings)
Silhouette in the embedding's own cosine space, on a 2,000-session subsample. **A guide, not the truth.** Values below about 0.25 suggest weak structure (Kaufman-Rousseeuw rule of thumb, verify before citing). Be suspicious of a curve that keeps *rising* with k: with many identical sessions, every extra cluster can simply isolate a group of duplicates (zero distance inside), which raises the silhouette without finding behaviour.

In [ ]:
sm = SemanticNLPModel(SessionRenderer("natural", rle=True), encoder=ENCODER).fit_texts(X)
t0 = time.perf_counter(); sm.fit_embeddings()
print(f"embedding: {time.perf_counter() - t0:.1f}s | {ENCODER.n_unique_:,} distinct texts of {ENCODER.n_texts_:,} | dim {sm.embeddings.shape[1]}")
rows = []
for k_ in range(2, 11):
    sm.cluster(k_, "sbert", random_state=SEED)
    rows.append({"k": k_, "silhouette (own space)": round(sm.silhouette_score(), 3),
                 "smallest cluster": f"{np.bincount(sm.labels_).min() / len(X):.1%}"})
ksweep = pd.DataFrame(rows).set_index("k"); display(ksweep.T)

## 6. Fit every approach at k = K
GSDMM gets only the upper bound `K_UB`, so its number of clusters is a result.

In [ ]:
fits, labels, timing = {}, {}, {}

def fit(name, builder):
    t0 = time.perf_counter(); obj = builder(); timing[name] = time.perf_counter() - t0
    fits[name] = obj; labels[name] = obj.labels_ if hasattr(obj, "labels_") else None
    print(f"{name:28s} k found {len(np.unique(labels[name])):2d} | {timing[name]:6.1f}s")

def b_stage2():
    t = TransitionModel(k=2).fit_kgrams(X); t.cluster(K, "kmeans", random_state=SEED); return t
def b_sbert():
    m = SemanticNLPModel(SessionRenderer("natural", rle=True), encoder=ENCODER).fit_texts(X); m.cluster(K, "sbert", random_state=SEED); return m
def b_gsdmm():
    m = SemanticNLPModel(term_mode="both").fit_texts(X); m.cluster(K_UB, "gsdmm", random_state=SEED); return m
def b_lda():
    m = SemanticNLPModel(term_mode="both").fit_texts(X); m.cluster(K, "lda", random_state=SEED); return m
def b_hdb():
    m = SemanticNLPModel(SessionRenderer("natural", rle=True), encoder=ENCODER).fit_texts(X)
    m.cluster(method="hdbscan", min_cluster_size=max(10, int(0.05 * len(X))), random_state=SEED); return m

fit("Stage 2 k-gram k=2 + k-means", b_stage2)
fit(f"embed ({ENCODER_NAME.split(' ')[0]}) + k-means", b_sbert)
fit("GSDMM (both terms)", b_gsdmm)
fit("LDA (both terms)", b_lda)
fit("embed + PCA + HDBSCAN (5%)", b_hdb)
labels = {k_: np.asarray(v) for k_, v in labels.items()}
pd.DataFrame({k_: pd.Series(np.sort(np.bincount(v))[::-1][:10]) for k_, v in labels.items()}).T.rename(columns=lambda c: f"size #{c+1}")

## 7. What are the clusters about?
`describe()`: size, mean length, distinctive terms (share of the cluster's sessions that contain the term vs share of all sessions) and the sessions closest to the centre. Shown for the SBERT/embedding route and for GSDMM; change the key to look at another.

In [ ]:
def show(sm_, top=5, ex=1, max_clusters=8):
    for c in sm_.describe(top_n=top, n_examples=ex)[:max_clusters]:
        terms = "; ".join(f"{t['term']} ({t['in_cluster']:.0%} vs {t['overall']:.0%})" for t in c["top_terms"])
        print(f"cluster {c['cluster']}  n={c['size']:,} ({c['share']:.0%})  mean len {c['mean_length']:.1f}\n   distinctive: {terms}\n   example    : {c['example_texts'][0][:200]}")
print("=== embeddings + k-means ==="); show(fits[f"embed ({ENCODER_NAME.split(' ')[0]}) + k-means"])

In [ ]:
print("=== GSDMM ==="); show(fits["GSDMM (both terms)"])

## 8. Judging the clusterings without ground truth
### 8.1 Predictive validity and the random-partition null
`eta^2` = share of the variance of `converted` explained by the clusters (0 = none, 1 = clusters are purely converting or not). It rises with the number of clusters even for random labels, so each value is compared with **200 random partitions of the same sizes**.

In [ ]:
def eta_squared(lab, y):
    tot = ((y - y.mean()) ** 2).sum()
    if tot == 0: return np.nan
    return float(sum((lab == c).sum() * (y[lab == c].mean() - y.mean()) ** 2 for c in np.unique(lab)) / tot)

rng_p = np.random.default_rng(1)
rows = []
for name, lab in labels.items():
    obs = eta_squared(lab, y)
    null = np.array([eta_squared(rng_p.permutation(lab), y) for _ in range(200)])
    rows.append({"approach": name, "clusters": len(np.unique(lab)), "eta^2 (observed)": round(obs, 3),
                 "random partitions: mean": round(null.mean(), 4), "random: 95th pct": round(np.percentile(null, 95), 4),
                 "beats chance": bool(obs > np.percentile(null, 95))})
pv = pd.DataFrame(rows).set_index("approach"); display(pv)

### 8.2 Conversion by cluster (one approach, as a table a manager can read)

In [ ]:
name = f"embed ({ENCODER_NAME.split(' ')[0]}) + k-means"
tab = pd.DataFrame({"cluster": labels[name], "converted": y}).groupby("cluster")["converted"].agg(["size", "mean"])
tab.columns = ["sessions", "conversion rate"]; tab["conversion rate"] = tab["conversion rate"].map("{:.1%}".format)
print("overall conversion:", f"{y.mean():.1%}"); display(tab)

### 8.3 Silhouette in a common reference space, stability and agreement
The common reference is k-gram cosine (k=2), on one 2,000-session subsample: every clustering is re-scored in the same geometry. Each reference favours its own family, so the Stage 2 row is expected to look good here; read the *differences*, not the level. Stability = ARI between the same method run with two seeds.

**Only the common-space column is comparable across rows.** The own-space column is shown for completeness: it lives in a different geometry per method, and for LDA it is inflated by construction (its clusters are the argmax of the very vectors the silhouette is computed on). Stage 2 has no own-space value because that would need the N x N matrix.

In [ ]:
sub = np.random.default_rng(7).choice(len(X), size=min(2000, len(X)), replace=False)
Xs = [X[i] for i in sub]
ref = TransitionModel(k=2).fit_kgrams(Xs); ref.fit_distance_matrix()

def common_silhouette(lab):
    ls = lab[sub]
    if len(np.unique(ls)) < 2: return np.nan
    ref.assign_labels(ls); return ref.silhouette_score()

def reseed(name):
    if name.startswith("Stage 2"):
        t = TransitionModel(k=2).fit_kgrams(X); t.cluster(K, "kmeans", random_state=SEED + 1); return t.labels_
    if name.startswith("embed (") :
        m = fits[name]; m.cluster(K, "sbert", random_state=SEED + 1); out = m.labels_.copy(); m.cluster(K, "sbert", random_state=SEED); return out
    if name.startswith("GSDMM"):
        m = SemanticNLPModel(term_mode="both").fit_texts(X); m.cluster(K_UB, "gsdmm", random_state=SEED + 1); return m.labels_
    if name.startswith("LDA"):
        m = SemanticNLPModel(term_mode="both").fit_texts(X); m.cluster(K, "lda", random_state=SEED + 1); return m.labels_
    return None

rows = []
for name, lab in labels.items():
    own = np.nan
    obj = fits[name]
    if isinstance(obj, SemanticNLPModel):
        try: own = obj.silhouette_score()
        except ValueError: pass
    else:
        own = None
    other = reseed(name)
    rows.append({"approach": name, "clusters": len(np.unique(lab)),
                 "silhouette (own space)": None if own is None else round(own, 3),
                 "silhouette (common: k-gram cosine)": round(common_silhouette(lab), 3),
                 "seed stability (ARI)": None if other is None else round(adjusted_rand_index(lab, other), 3),
                 "fit time (s)": round(timing[name], 1)})
display(pd.DataFrame(rows).set_index("approach"))

names = list(labels)
agree = pd.DataFrame([[adjusted_rand_index(labels[a], labels[b]) for b in names] for a in names], index=names, columns=names).round(2)
print("agreement between approaches (ARI):"); display(agree)

## 9. Shared Markov profile and cost
### 9.1 The same behaviour profile on any clustering
`profile()` fits Stage 2's chains on the semantic clusters: where each cluster starts, where it drops off, how likely it is to reach a cart event. It is fitted on **actions** (`event_map`), because the category detail is already described by the distinctive terms of section 7.

In [ ]:
m = fits[f"embed ({ENCODER_NAME.split(' ')[0]}) + k-means"]
# Profile at ACTION level (view / cart): with category-level tokens the chain would have thousands of
# states and "reaching a cart" would be spread over hundreds of different cart tokens.
to_action = lambda s: [r0.split_token(e)[0] or e for e in s]
prof = m.profile(event_map=to_action)
target = next((t for t in ("cart", "addtocart") if t in prof.states), None)
rows = []
for c in range(m.n_clusters_found_):
    d = prof.dropoff_rates(c); e = prof.entry_probabilities(c)
    rows.append({"cluster": c, "sessions": int((m.labels_ == c).sum()),
                 "top entry": max(e, key=e.get), "top drop-off": max(d, key=d.get),
                 "P(reach a cart event)": round(prof.conversion_probability(target, c)[START], 2) if target else None})
display(pd.DataFrame(rows).set_index("cluster"))
print("representative action paths (categories are described in section 7):")
for c in range(min(6, m.n_clusters_found_)):
    print(f"  cluster {c}: {' > '.join(prof.get_representative_path(c)[0])[:160]}")

### 9.2 Scalability on real sessions
Times for growing random subsets. **Read the distinct-text ratio**: real logs repeat the same short sessions, and the encoder encodes each distinct text once, so its cost depends on that ratio, not only on N. The last row repeats the largest encode with `dedup=False` to show the uncached cost. With the stand-in, the embed column measures hashing, not SBERT.

In [ ]:
rows = []
for n in (2000, 5000, 10000, 20000):
    if n > len(X): break
    sub_n = [X[i] for i in np.random.default_rng(n).choice(len(X), n, replace=False)]
    m_ = SemanticNLPModel(SessionRenderer("natural", rle=True), encoder=ENCODER)
    t0 = time.perf_counter(); m_.fit_texts(sub_n); t_r = time.perf_counter() - t0
    t0 = time.perf_counter(); m_.fit_embeddings(); t_e = time.perf_counter() - t0
    t0 = time.perf_counter(); m_.cluster(K, "sbert", random_state=0); t_k = time.perf_counter() - t0
    g = SemanticNLPModel(term_mode="both").fit_texts(sub_n)
    t0 = time.perf_counter(); g.cluster(K_UB, "gsdmm", random_state=0, n_iter=15); t_g = time.perf_counter() - t0
    t0 = time.perf_counter(); t2 = TransitionModel(k=2).fit_kgrams(sub_n); t2.cluster(K, "kmeans", random_state=0); t_2 = time.perf_counter() - t0
    rows.append({"N": n, "distinct texts": f"{ENCODER.n_unique_ / n:.0%}", "render+terms": t_r, "embed": t_e, "k-means": t_k,
                 "GSDMM (15 sweeps)": t_g, "Stage 2 k-gram + k-means": t_2})
scal = pd.DataFrame(rows).set_index("N"); display(scal.round(2))

nn = len(sub_n); raw = SBERTEncoder(encoder=ENCODER.load(), dedup=False)
t0 = time.perf_counter(); raw.encode(m_.texts); print(f"N={nn:,}: encode WITHOUT dedup {time.perf_counter() - t0:.2f}s vs with dedup {t_e:.2f}s "
                                                       f"({ENCODER.n_unique_:,} distinct of {nn:,} texts)")

## 10. What to report from this notebook
For every approach: number of clusters found, `eta^2` against its random-partition 95th percentile, own and common silhouette, seed stability, fit time, the encoder name and rendering, N, and the distinct-text ratio. Then, in words: what each cluster is (section 7) and whether the approaches agree (8.3).

**Interpretation rules**
* A clustering that does not beat the random-partition 95th percentile in 8.1 describes nothing about conversion, however good its silhouette.
* Low seed stability means the partition is not a property of the data.
* If SBERT does not beat the stand-in encoder on 8.1 and 8.3 (re-run with `USE_SBERT = False` to get it), word meaning added nothing here.
* HDBSCAN's noise cluster is part of the partition above; report results with and without it (`sm.noise_mask_`).
* Results are for one random sample; repeat with other `SEED` values before drawing conclusions.